In [ ]:
#| default_exp skill
#| export
import re

from nbskill.edit import edit_notebook
from nbskill.execute import exec_nb as _exec_nb
from nbskill.foundation import generated_owner
from nbskill.knowledge import reference_query as _reference_query
from nbskill.read import context
from nbskill.review import diff_nb as _diff_nb, style_report as _style_report

In [ ]:
from pathlib import Path
from nbskill import skill
from tempfile import gettempdir
from fastcore.nbio import mk_cell, read_nb
from nbskill.foundation import write_demo_notebook

# Native notebook Pyskill

> Core notebook operations for coding agents.

#| export
`nbskill.skill` is the authoritative notebook workflow for an agent working with nbdev source. It supplies operations for preparing a change, editing cells, and proving the result.

Standalone instructions only route notebook-owned work here. Use it after aai-coding identifies the source; it does not replace the general coding workflow.

#| export
## Core workflow

For a nontrivial task, start with `prepare_change`: it combines compact source context with local prior art. Use `edit_notebook` for the one structured mutation, then `verify_change` to prove the code diff, focused execution, and changed-source diagnostics. Do not mutate raw `.ipynb` JSON or a generated module.

#| export
## Route source files

For a Python file, `generated_owner(path)` returns its notebook source or `None`. Use this Pyskill only for the returned notebook; every other file stays on the normal aai-coding path.

In [ ]:
#| exportd
summary = context("nbs/01_read.ipynb#context", scope="nbs", view="summary", verbose=False)
assert summary["kind"] == "context"

#| export
## Edit and review

Pass one small, explicit operation to `edit_notebook`. It validates and exports the notebook. `verify_change` runs only the affected scope without writing outputs, then returns the code diff and changed-source diagnostics.

```python
from nbskill.skill import *

prepare_change("nbs/02_edit.ipynb", "find the existing edit pattern")
edit_notebook("nbs/02_edit.ipynb", [edit])
proof = verify_change("nbs/02_edit.ipynb")
```

#| export
`verify_change` includes diagnostics only for changed source. It complements the behavior check and code-cell diff instead of replacing either.

## Prepare and prove a change

`prepare_change` gathers the notebook's compact context and local implementation prior art before a nontrivial change. It deliberately keeps reference lookup offline, so planning does not silently fetch a dependency.

`verify_change` then reviews the code-cell diff, runs the affected scope without writing outputs, and reports diagnostics only for changed source. These two operations group the decisions around a change; `edit_notebook` remains the one structured mutation step.

In [ ]:
#| exporti
def _changed_execution_target(diff, up2id=None):
    "Choose an explicit execution target or the last changed code cell."
    if up2id is not None: return up2id
    changed = re.findall(r"^--- code cell (.+?) ---$", diff, flags=re.M)
    return changed[-1] if changed else None

In [ ]:
#| export
def prepare_change(
    path: str,  # Notebook path to inspect
    question: str,  # Implementation question for prior-art lookup
    scope: str = ".",  # Workspace used for context and local prior art
    top_k: int = 5,  # Number of prior-art hits to retain
    reference_path: str | None = None,  # Optional local reference index
):
    "Gather notebook context and local prior art before a change."
    return dict(
        context=context(path, scope=scope, view="summary", verbose=False),
        references=_reference_query(
            question, top_k=top_k, current_repo=scope, path=reference_path,
            include_local=True, allow_download=False,
        ),
    )

In [ ]:
#| export
def verify_change(
    path: str,  # Explicit affected notebook
    up2id: int | str | None = None,  # Optional focused execution target
    ref_a: str | None = "HEAD",  # First diff reference
    ref_b: str | None = None,  # Second diff reference
    timeout: int = 30,  # Per-cell execution timeout
    allow_new: bool = False,  # Permit unapproved cells in a disposable validation
):
    "Review a notebook diff, focused execution, and changed-source diagnostics."
    diff = _diff_nb(path, ref_a=ref_a, ref_b=ref_b)
    up2id = _changed_execution_target(diff, up2id)
    execution = _exec_nb(
        path, up2id=up2id, timeout=timeout, allow_new=allow_new,
        check_only=True, show_output=False,
    )
    diagnostics = _style_report(path, changed_only=True, ref_a=ref_a, ref_b=ref_b)
    return dict(path=str(path), up2id=up2id, diff=diff, execution=execution, diagnostics=diagnostics)

In [ ]:
#| export

__all__ = ["context", "generated_owner", "prepare_change", "edit_notebook", "verify_change"]

### Direct workflow check

This test uses the public Pyskill to make a structured change to a fresh notebook. It protects the native path from quietly depending on MCP.

In [ ]:
#| hide



with write_demo_notebook(
    "14_pyskill_workflow.ipynb",
    base=gettempdir(),
    cells=[
        mk_cell("## Answer", cell_type="markdown"),
        mk_cell("answer = 41", cell_type="code"),
        mk_cell("assert answer == 42", cell_type="code"),
    ],
) as path:
    inspected = skill.context(str(path), scope=str(path), view="summary", verbose=False)
    assert inspected["kind"] == "context"

    answer = read_nb(path).cells[1]
    result = skill.edit_notebook(
        str(path),
        [dict(op="replace_text", cell_id=answer.id, old="answer = 41", new="answer = 42")],
        auto_feedback=False,
    )
    assert result["changed"]

### Native change proof

verify_change must review an actual code-cell change, execute only the changed scope, and report diagnostics for changed source.

In [ ]:
#| hide
with write_demo_notebook(
    "14_pyskill_verify_change.ipynb",
    base=gettempdir(),
    cells=[
        mk_cell("answer = 41", cell_type="code"),
        mk_cell("assert answer == 42", cell_type="code"),
    ],
) as path:
    answer = read_nb(path).cells[0]
    skill.edit_notebook(
        str(path),
        [dict(op="replace_text", cell_id=answer.id, old="answer = 41", new="answer = 42")],
        auto_feedback=False,
    )
    proof = skill.verify_change(str(path), ref_a=None, ref_b=None, allow_new=True, timeout=5)
    assert proof["execution"] == path
    assert proof["up2id"]
    assert "+answer = 42" in proof["diff"]
    assert proof["diagnostics"]["summary"]["changed_only"]

In [ ]:
#| hide

assert "`verify_change` includes diagnostics" in skill.__doc__

module_text = Path(skill.__file__).read_text()
assert "summary = context(" in skill.__doc__
assert "Pass one small, explicit operation to `edit_notebook`." in skill.__doc__
assert "../nbs/14_pyskill.ipynb" in module_text
assert set(skill.__all__) == {"context", "generated_owner", "prepare_change", "edit_notebook", "verify_change"}
assert "For a nontrivial task, start with `prepare_change`" in skill.__doc__
assert "every other file stays on the normal aai-coding path" in skill.__doc__
generated = Path.cwd().parent / "nbskill" / "skill.py"
assert skill.generated_owner(generated).name == "14_pyskill.ipynb"

handwritten = Path(gettempdir()) / "14_pyskill_handwritten.py"
handwritten.write_text("value = 1\n")
try: assert skill.generated_owner(handwritten) is None
finally: handwritten.unlink()